In [1]:
import numpy as np
import pandas as pd
import os, sys

from collections import Counter
import matplotlib.pyplot as plt
import pickle
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [2]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from torch.utils.data import Dataset
from transformers import AutoModelForSequenceClassification
from transformers import Trainer, TrainingArguments

In [3]:
import torch
import torch.nn.functional as F

In [4]:
from transformers import AutoTokenizer

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
gdrive_path = "/content/drive/MyDrive/TheoryofML_GroupProject/externalities"

In [7]:
!ls "/content/drive/MyDrive/TheoryofML_GroupProject/externalities"

gutenberg  gutenberg-analysis


In [8]:
## path to the downloaded gutenberg corpus
path_gutenberg = os.path.join(gdrive_path,'gutenberg')

In [9]:
path_gutenberg

'/content/drive/MyDrive/TheoryofML_GroupProject/externalities/gutenberg'

In [10]:
# src_dir = '/Users/nikitaparulekar/GitRepos/gutenberg-analysis/src'
src_dir = os.path.join(gdrive_path,'gutenberg-analysis','src')
sys.path.append(src_dir)
from data_io import get_book

In [11]:
# sys.path.append(os.path.join(path_gutenberg, 'src'))
# from metaquery import meta_query
# mq = meta_query(path=os.path.join(path_gutenberg, 'metadata', 'metadata.csv'))


In [12]:
#mq.df['author'].unique()

In [13]:
print(os.getcwd())

/content


In [14]:
# get the updated csv
# data_path = os.path.join(os.pardir, 'sample_dataset')
train_path = os.path.join(gdrive_path,os.pardir, 'final_train.csv')
test_path = os.path.join(gdrive_path,os.pardir, 'final_test.csv')
val_path = os.path.join(gdrive_path,os.pardir, 'final_val.csv')

In [15]:
train_meta_df = pd.read_csv(train_path)
test_meta_df = pd.read_csv(test_path)
val_meta_df = pd.read_csv(val_path)

In [16]:
val_meta_df['author'].nunique(),test_meta_df['author'].nunique(),train_meta_df['author'].nunique()

(80, 80, 80)

In [17]:
test_meta_df['author'].value_counts().unique()

array([3])

In [18]:
# def split_test_validation(test_meta_df, val_samples_per_author=3):
#     author_list = test_meta_df['author'].unique()
#     val_indices = []

#     for author in author_list:
#         # Get indices for the current author
#         author_indices = test_meta_df[test_meta_df['author'] == author].index

#         # Sample validation indices for this author
#         author_val_indices = np.random.choice(author_indices, val_samples_per_author, replace=False)
#         val_indices.extend(author_val_indices)

#     # Create validation dataframe
#     val_df = test_meta_df.loc[val_indices]

#     # Remove validation samples from test dataframe
#     updated_test_df = test_meta_df.drop(val_indices)

#     return updated_test_df, val_df

In [19]:
# test_meta_df, val_meta_df = split_test_validation(test_meta_df)

In [20]:
# test_meta_df.loc[test_meta_df['author'] == 'Doyle, Arthur Conan']

In [21]:
# val_meta_df.loc[val_meta_df['author'] == 'Doyle, Arthur Conan']

In [22]:
# test_meta_df['author'].value_counts()

In [23]:
# val_meta_df['author'].value_counts()

In [24]:
# val_path = os.path.join(os.pardir,'sample_dataset', 'final_val.csv')
# val_meta_df.to_csv(val_path)
# test_path = os.path.join(os.pardir,'sample_dataset', 'final_test.csv')
# test_meta_df.to_csv(test_path)


In [25]:
train_meta_df.head()

,Unnamed: 0,id,title,author,authoryearofbirth,authoryearofdeath,language,downloads,subjects
0,2439,PG12810,"Uncle Sam's Boys with Pershing's Troops: Or, D...","Hancock, H. Irving (Harrie Irving)",1868.0,1922.0,['en'],78,"{'World War, 1914-1918 -- Juvenile fiction', '..."
1,2446,PG12819,"Dick Prescott's Second Year at West Point: Or,...","Hancock, H. Irving (Harrie Irving)",1868.0,1922.0,['en'],94,{'United States Military Academy -- Juvenile f...
2,25920,PG40605,"The Motor Boat Club at Nantucket; or, The Myst...","Hancock, H. Irving (Harrie Irving)",1868.0,1922.0,['en'],189,"{'Motorboats -- Juvenile fiction', 'Nantucket ..."
3,55435,PG8153,"The Young Engineers in Arizona; or, Laying Tra...","Hancock, H. Irving (Harrie Irving)",1868.0,1922.0,['en'],190,"{'Civil engineers -- Fiction', 'Arizona -- Fic..."
4,32899,PG48863,"The Motor Boat Club off Long Island; or, A Dar...","Hancock, H. Irving (Harrie Irving)",1868.0,1922.0,['en'],85,"{'Motorboats -- Juvenile fiction', 'Long Islan..."


In [26]:
# get the full test for each id-author (how to store?)
# tokenize the text
# pretrain transformer with txt classification

In [27]:
train_meta_df['author'].nunique()

80

In [28]:
# train_meta_df['author'].unique()

In [29]:
num_authors = train_meta_df['author'].nunique()

# Tokenization

In [30]:
# tokenize the books
# get 20 chunks of 512 tokens each, ~uniformly through each book

In [31]:
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [32]:
tokenizer

DistilBertTokenizerFast(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [33]:
# encoded_text = tokenizer("I am hungry, and want 26 Birria tacos ")

In [34]:
# encoded_text["input_ids"][0]

In [35]:
# tokens = tokenizer.convert_ids_to_tokens(encoded_text.input_ids)
# print(tokens)

In [36]:
# print(tokenizer.convert_tokens_to_string(tokens))

In [37]:
# train_meta_df.head()

In [38]:
# def tokenize(batch):
#     '''keep last n tokens, ignore '''
#     return tokenizer(batch["text"], padding=True, truncation=True,truncation_side="left")

In [39]:
# train_meta_df['author'].unique()

In [40]:
# book_1 = get_book('PG12810', level='text')

In [41]:
# book_1[-512:]

# Approach 1: Chunking and Aggregating

In [42]:
# def get_token_samples(num_samples_per_book,meta_df):
#     ''''Extract num_samples_per_book of max size truncated from left'''
#     # for each author
#     for author in meta_df['author'].unique():
#         author_df = meta_df.loc[meta_df['author'] == author]
#         # for each book id , get full book
#         for id in author_df['id'].unique():
#             book_str = get_book(id, level='text')
#             # tokenize full text (no truncation)

#             # for sample in num_samples_per_book
#             for sample_num in range(num_samples_per_book):
#                 # take last 512 * num_samples_per_book of token_list
#                 # partition that into num_samples_per_book array
#                 #store in a dict of author:nested array of all samples

In [43]:
# dummy_meta_df = train_meta_df.loc[(train_meta_df['id'] == 'PG63902') | (train_meta_df['id'] == 'PG41628')]

In [44]:
# dummy_meta_df

In [45]:
#train_meta_df.head(45)

In [46]:
# dummy_val_meta_df = train_meta_df.loc[(train_meta_df['id'] == 'PG12690') | (train_meta_df['id'] == 'PG63142')]

In [47]:
def get_token_samples(path_gutenberg,num_samples_per_book, meta_df, tokenizer, max_length=512):
    '''Extract num_samples_per_book of max size truncated from left'''
    # Dictionary to store samples by author
    author_samples = {}
    author_attention_masks = {}

    # for each author
    for author in meta_df['author'].unique():
        author_df = meta_df.loc[meta_df['author'] == author]
        author_samples[author] = []
        author_attention_masks[author] = []

        # for each book id, get full book
        for id in author_df['id'].unique():

            try:
                book_str = get_book(id,path_gutenberg = path_gutenberg, level='text')
            except FileNotFoundError:
                print(f"{id} missing for author {author}")
                continue


            # tokenize full text (no truncation)
            # it returns a dict with input_ids & attention_mask
            tokens_dict = tokenizer(book_str, truncation=False, return_tensors="pt")
            tokens = tokens_dict["input_ids"][0]
            masks = tokens_dict["attention_mask"][0]

            # calculate total tokens needed
            total_tokens_needed = num_samples_per_book * max_length

            # If the book is too short, skip or pad as needed
            if len(tokens) < total_tokens_needed:
                # Option 1: Skip short books
                # continue

                # Option 2: Use what we have and pad the rest with repetition
                while len(tokens) < total_tokens_needed:
                    tokens = torch.cat([tokens, tokens])
                    masks = torch.cat([masks, masks])

            # Take the last total_tokens_needed tokens
            last_n_tokens = tokens[-total_tokens_needed:]
            last_n_masks = masks[-total_tokens_needed:]

            # Partition into num_samples_per_book samples
            for sample_num in range(num_samples_per_book):
                start_idx = sample_num * max_length
                end_idx = start_idx + max_length

                sample = last_n_tokens[start_idx:end_idx].tolist()
                attention_mask = last_n_masks[start_idx:end_idx].tolist()

                # Store in the author's samples list
                author_samples[author].append(sample)
                author_attention_masks[author].append(attention_mask)

    # Convert to format ready for dataset creation
    all_samples = []
    all_labels = []
    all_attention_masks = []
    label_to_id = {author: idx for idx, author in enumerate(author_samples.keys())}

    for author, samples in author_samples.items():
        label_id = label_to_id[author]
        for i,sample in enumerate(samples):
            all_samples.append(sample)
            all_attention_masks.append(author_attention_masks[author][i])
            all_labels.append(label_id)

    return all_samples, all_attention_masks, all_labels, label_to_id

In [48]:
# dummy_meta_df

In [49]:
num_samples_per_book = 5
max_length = 512 #DistilBert specific

In [50]:
path_gutenberg

'/content/drive/MyDrive/TheoryofML_GroupProject/externalities/gutenberg'

In [51]:
train_samples, train_attention_masks, train_labels, train_label_to_id = get_token_samples(
                                                       path_gutenberg,
                                                      num_samples_per_book,
                                                       train_meta_df,
                                                       tokenizer,
                                                       max_length=max_length)

Token indices sequence length is longer than the specified maximum sequence length for this model (62762 > 512). Running this sequence through the model will result in indexing errors


PG3748 missing for author Verne, Jules
PG75746 missing for author Stratemeyer, Edward
PG75629 missing for author Belloc, Hilaire
PG75642 missing for author Wells, Carolyn
PG75665 missing for author Meade, L. T.
PG75765 missing for author Blanchard, Amy Ella
PG75758 missing for author Leinster, Murray
PG60311 missing for author Westerman, Percy F. (Percy Francis)
PG75760 missing for author Le Queux, William


In [52]:
import sys

In [53]:
print(f"train_samples: {sys.getsizeof(train_samples)} bytes")
print(f"train_attention_masks: {sys.getsizeof(train_attention_masks)} bytes")
print(f"train_labels: {sys.getsizeof(train_labels)} bytes")
print(f"train_label_to_id: {sys.getsizeof(train_label_to_id)} bytes")

train_samples: 85176 bytes
train_attention_masks: 85176 bytes
train_labels: 85176 bytes
train_label_to_id: 1584 bytes


In [55]:
val_samples,val_attention_masks, val_labels, val_label_to_id = get_token_samples(path_gutenberg,
                                                                                 num_samples_per_book,
                                                       val_meta_df,
                                                       tokenizer,
                                                       max_length=max_length)

In [56]:
# import pickle

In [57]:
# save it as pkl
# path = "/Users/nikitaparulekar/Desktop/MachineLearningTheory/FinalProject/saved_data/approach1/tokenized_data"



In [58]:
# all_samples, all_attention_masks, all_labels, label_to_id = get_token_samples(3,
#                                                        dummy_meta_df,
#                                                        tokenizer,
#                                                        max_length=512)

In [59]:
# val_samples,val_attention_mask, val_labels, val_label_to_id = get_token_samples(3,
#                                                        dummy_val_meta_df,
#                                                        tokenizer,
#                                                        max_length=512)

In [60]:
# check sample

In [61]:
# check sample length
# for i in range(len(all_samples)):
#     print(len(all_samples[i]))


In [ ]:
# convert above into a Pytorch Dataset object



In [63]:
class CustomTextDataset(Dataset):
    def __init__(self, labels, samples, attention_masks, transform=None, target_transform=None):
        self.labels = labels
        self.samples = samples
        self.attention_masks = attention_masks
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Get the token sequence and attention mask at the given index
        sample = self.samples[idx]
        attention_mask = self.attention_masks[idx]

        # Convert to tensors if not already tensors
        if not isinstance(sample, torch.Tensor):
            sample = torch.tensor(sample)
        if not isinstance(attention_mask, torch.Tensor):
            attention_mask = torch.tensor(attention_mask)

        # Get the corresponding label
        label = self.labels[idx]

        if self.transform:
            sample = self.transform(sample)
        if self.target_transform:
            label = self.target_transform(label)

        # Return in the format expected by HuggingFace Trainer
        return {
            "input_ids": sample,
            "attention_mask": attention_mask,
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [64]:
train_dataset =  CustomTextDataset(
    labels=train_labels,
    samples=train_samples,
    attention_masks=train_attention_masks
)

val_dataset = CustomTextDataset(
    labels=val_labels,
    samples=val_samples,
    attention_masks=val_attention_masks)

In [65]:
val_dataset

In [66]:
len(train_dataset)

9555

In [67]:
# from datasets import load_dataset

In [68]:
# emotions = load_dataset("emotion")

In [69]:
# emotions["train"][:2]


In [70]:
# type(emotions["train"])

In [71]:
## performance metrics

In [72]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, average="weighted") # dont really need weightd since no class imbal in chuning and agreation methd. but could be there in othr methods due to diffrent txt length
    recall = recall_score(labels, preds, average="weighted")
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [73]:
## loading the pretrained model

In [74]:
model_ckpt = "distilbert-base-uncased"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [75]:
# os.environ["PYTORCH_ENABLE_MPS_DEVICE"] = "0"


In [76]:
device

device(type='cuda')

In [77]:
ab

NameError: name 'ab' is not defined

In [80]:
num_labels = train_meta_df['author'].nunique()
model = (AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=num_labels) .to(device))

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
## training the model (logged in via terminal)

In [81]:
batch_size = 64 #CHANGE This to 64 later
logging_steps = len(train_dataset) // batch_size
model_name = f"{model_ckpt}-finetuned-author-classification"
training_args = TrainingArguments(output_dir=model_name,
                                  num_train_epochs=2,
                                  learning_rate=2e-5,
                                  per_device_train_batch_size=batch_size,
                                  per_device_eval_batch_size=batch_size,
                                  weight_decay=0.01,
                                  eval_strategy="epoch",
                                  disable_tqdm=False,
                                  logging_steps=logging_steps,
                                  push_to_hub=False, ## change this later
                                  log_level="error"
                                #   dataloader_pin_memory=False,     # Disable pinned memory
                                  # no_cuda=True        # Explicitly disable CUDA/GPU
)

In [82]:
trainer = Trainer(model=model,
                  args=training_args,
                  compute_metrics=compute_metrics,
                  train_dataset=train_dataset,
                  eval_dataset=val_dataset,
                  tokenizer=tokenizer)


<ipython-input-82-9f1a6c505153>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model,


In [83]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ny-parulekar (ny-parulekar-johns-hopkins-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,4.202000,4.349046,0.017500,0.013562,0.016479,0.017500
2,3.816400,4.361479,0.024167,0.017947,0.016222,0.024167


/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


TrainOutput(global_step=300, training_loss=4.008280413945516, metrics={'train_runtime': 1206.0408, 'train_samples_per_second': 15.845, 'train_steps_per_second': 0.249, 'total_flos': 2534973286809600.0, 'train_loss': 4.008280413945516, 'epoch': 2.0})

In [84]:
model_save_path = f"/content/drive/MyDrive/TheoryofML_GroupProject/{model_name}"
trainer.save_model(model_save_path)